In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dirty-dataset-to-practice-data-cleaning/my_file (1).csv


# Dataset Overview

### At begin, lets look at some data values/types and possible anomalies.

In [2]:
df_raw = pd.read_csv('/kaggle/input/dirty-dataset-to-practice-data-cleaning/my_file (1).csv')

print(f"Rows: {df_raw.shape[0]}\nColumns: {df_raw.shape[1]}")

Rows: 20
Columns: 11


In [3]:
df = df_raw.copy()

df.sample(5, random_state=42)

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
17,18,NaN,NaN,"$169,800,000","$207,046,755",Madonna,Rebel Heart Tour,2015–2016,82,"$2,070,732",[4]
15,16,NaN,NaN,"$184,000,000","$227,452,347",Pink,The Truth About Love Tour,2013–2014,142,"$1,295,775",[22]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
8,9,NaN,NaN,"$256,084,556","$312,258,401",Beyoncé,The Formation World Tour,2016,49,"$5,226,215",[13]


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Rank                              20 non-null     int64 
 1   Peak                              9 non-null      object
 2   All Time Peak                     6 non-null      object
 3   Actual gross                      20 non-null     object
 4   Adjusted gross (in 2022 dollars)  20 non-null     object
 5   Artist                            20 non-null     object
 6   Tour title                        20 non-null     object
 7   Year(s)                           20 non-null     object
 8   Shows                             20 non-null     int64 
 9   Average gross                     20 non-null     object
 10  Ref.                              20 non-null     object
dtypes: int64(2), object(9)
memory usage: 1.8+ KB


# Header Cleaning

### First, cheking if the header values are in the correct format

In [5]:
print(f"Before: \n{df.columns.to_list()}")

df.columns = (df.columns
              .str.lower()
              .str.replace(" ", "_")
              .str.replace("\xa0", "_")
              .str.replace(r"[^\w]", "", regex=True))

print(f"\nAfter: \n{df.columns.to_list()}")

Before: 
['Rank', 'Peak', 'All Time Peak', 'Actual\xa0gross', 'Adjusted\xa0gross (in 2022 dollars)', 'Artist', 'Tour title', 'Year(s)', 'Shows', 'Average gross', 'Ref.']

After: 
['rank', 'peak', 'all_time_peak', 'actual_gross', 'adjusted_gross_in_2022_dollars', 'artist', 'tour_title', 'years', 'shows', 'average_gross', 'ref']


# Handling Values

### Here we will drop unnecessary columns and deal with missing or incorrectly formatted values.

### The dataset was generated by web scraping a wikipedia table. The "ref." column indicates the wikipedia reference URL, that we didnt get during the scraping process, so this column is unnecessary and irrelevant at this time.

In [6]:
df[["ref"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
ref,[1],[3],[6],[7],[8],[9],[11],[12],[13],[14],[15][16],[18],[19],[20],[5],[22],[d],[4],[25],[26]


In [7]:
df.drop("ref", axis=1, inplace=True)

### The columns "peak" and "all-time_peak" have many unavailable values, we can easily search for this information on the internet, because table only has 20 rows. Probably, in a real world scenario, this wouldnt be possible. In this context, lets just drop both columns and move on.

In [8]:
df[["peak", "all_time_peak"]].isna().sum()

peak             11
all_time_peak    14
dtype: int64

In [9]:
df.drop(["peak", "all_time_peak"], axis=1, inplace=True)

### Analyzing the columns "actual_gross", "adjusted_gross_in_2022_dollars", and "average_gross", all refer to monetary values. To prepare this data for future analysis and transformations, we need to keep only the numbers.

In [10]:
df[["actual_gross", "adjusted_gross_in_2022_dollars", "average_gross"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
actual_gross,"$780,000,000","$579,800,000","$411,000,000","$397,300,000","$345,675,146","$305,158,363","$280,000,000","$257,600,000","$256,084,556","$250,400,000","$229,100,000[b]","$227,400,000","$204,000,000","$200,000,000","$194,000,000","$184,000,000","$170,000,000","$169,800,000","$167,700,000[e]","$150,000,000"
adjusted_gross_in_2022_dollars,"$780,000,000","$579,800,000","$560,622,615","$454,751,555","$402,844,849","$388,978,496","$381,932,682","$257,600,000","$312,258,401","$309,141,878","$283,202,896","$295,301,479","$251,856,802","$299,676,265","$281,617,035","$227,452,347","$213,568,571","$207,046,755","$204,486,106","$185,423,109"
average_gross,"$13,928,571","$10,353,571","$4,835,294","$2,546,795","$6,522,173","$3,467,709","$2,137,405","$6,282,927","$5,226,215","$2,945,882","$1,735,606","$1,118,227","$1,350,993","$615,385","$3,233,333","$1,295,775","$1,734,694","$2,070,732","$1,385,950","$1,744,186"


In [11]:
cols_money = ["actual_gross", "adjusted_gross_in_2022_dollars", "average_gross"]

df[cols_money ] = (df[cols_money ]
                     .replace(r"[^0-9.]", "", regex=True)
                     .astype(float))

In [12]:
df[["actual_gross", "adjusted_gross_in_2022_dollars", "average_gross"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
actual_gross,780000000.0,579800000.0,411000000.0,397300000.0,345675146.0,305158363.0,280000000.0,257600000.0,256084556.0,250400000.0,229100000.0,227400000.0,204000000.0,200000000.0,194000000.0,184000000.0,170000000.0,169800000.0,167700000.0,150000000.0
adjusted_gross_in_2022_dollars,780000000.0,579800000.0,560622615.0,454751555.0,402844849.0,388978496.0,381932682.0,257600000.0,312258401.0,309141878.0,283202896.0,295301479.0,251856802.0,299676265.0,281617035.0,227452347.0,213568571.0,207046755.0,204486106.0,185423109.0
average_gross,13928571.0,10353571.0,4835294.0,2546795.0,6522173.0,3467709.0,2137405.0,6282927.0,5226215.0,2945882.0,1735606.0,1118227.0,1350993.0,615385.0,3233333.0,1295775.0,1734694.0,2070732.0,1385950.0,1744186.0


### Following the same logic used for monetary values, the "tour_title" column only needs to contain A-Z, 0-9 and ("&", ":", ".") characters.

In [13]:
df[["tour_title"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
tour_title,The Eras Tour †,Renaissance World Tour,Sticky & Sweet Tour ‡[4][a],Beautiful Trauma World Tour,Reputation Stadium Tour,The MDNA Tour,Taking Chances World Tour,Summer Carnival †,The Formation World Tour,The 1989 World Tour,The Mrs. Carter Show World Tour,The Monster Ball Tour *,Prismatic World Tour,Living Proof: The Farewell Tour ‡[21][a],Confessions Tour,The Truth About Love Tour,Born This Way Ball,Rebel Heart Tour,Adele Live 2016,The Red Tour


In [14]:
df["tour_title"] = (df["tour_title"]
                    .astype(str)
                    .replace([r"\[.*\]",r"[^a-zA-Z0-9\s&.:]"], "", regex=True)
                    .str.strip())

In [15]:
df[["tour_title"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
tour_title,The Eras Tour,Renaissance World Tour,Sticky & Sweet Tour,Beautiful Trauma World Tour,Reputation Stadium Tour,The MDNA Tour,Taking Chances World Tour,Summer Carnival,The Formation World Tour,The 1989 World Tour,The Mrs. Carter Show World Tour,The Monster Ball Tour,Prismatic World Tour,Living Proof: The Farewell Tour,Confessions Tour,The Truth About Love Tour,Born This Way Ball,Rebel Heart Tour,Adele Live 2016,The Red Tour


# Feature Extraction

### The "years" column indicates the period (start year - end year) of the tour, but the values ​​are stored in string format, which makes it difficult to obtain the year information. An alternative would be to separate the years into different columns.

In [16]:
df[["years"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
years,2023–2024,2023,2008–2009,2018–2019,2018,2012,2008–2009,2023–2024,2016,2015,2013–2014,2009–2011,2014–2015,2002–2005,2006,2013–2014,2012–2013,2015–2016,2016–2017,2013–2014


In [17]:
df["start_year"] = (df["years"].str.extract(r"(\d{4})")).astype(int)
df["end_year"] = (df["years"].str.extract(r"(\d{4})$")).astype(int)

In [18]:
df[["start_year", "end_year"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
start_year,2023,2023,2008,2018,2018,2012,2008,2023,2016,2015,2013,2009,2014,2002,2006,2013,2012,2015,2016,2013
end_year,2024,2023,2009,2019,2018,2012,2009,2024,2016,2015,2014,2011,2015,2005,2006,2014,2013,2016,2017,2014


### The original "year(s)" column is now irrelevant, we could simply remove it, but lets store there the tour year count.

In [19]:
df["years"] = df["end_year"] - df["start_year"]

In [20]:
df[["years"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
years,1,0,1,1,0,0,1,1,0,0,1,2,1,3,0,1,1,1,1,1


# Overview

### Now lets compare the initial df with the current, after the changes.

In [21]:
df_raw.sample(5, random_state=42)

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
17,18,NaN,NaN,"$169,800,000","$207,046,755",Madonna,Rebel Heart Tour,2015–2016,82,"$2,070,732",[4]
15,16,NaN,NaN,"$184,000,000","$227,452,347",Pink,The Truth About Love Tour,2013–2014,142,"$1,295,775",[22]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
8,9,NaN,NaN,"$256,084,556","$312,258,401",Beyoncé,The Formation World Tour,2016,49,"$5,226,215",[13]


In [22]:
df.sample(5, random_state=42)

,rank,actual_gross,adjusted_gross_in_2022_dollars,artist,tour_title,years,shows,average_gross,start_year,end_year
0,1,780000000.0,780000000.0,Taylor Swift,The Eras Tour,1,56,13928571.0,2023,2024
17,18,169800000.0,207046755.0,Madonna,Rebel Heart Tour,1,82,2070732.0,2015,2016
15,16,184000000.0,227452347.0,Pink,The Truth About Love Tour,1,142,1295775.0,2013,2014
1,2,579800000.0,579800000.0,Beyoncé,Renaissance World Tour,0,56,10353571.0,2023,2023
8,9,256084556.0,312258401.0,Beyoncé,The Formation World Tour,0,49,5226215.0,2016,2016


In [23]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Rank                              20 non-null     int64 
 1   Peak                              9 non-null      object
 2   All Time Peak                     6 non-null      object
 3   Actual gross                      20 non-null     object
 4   Adjusted gross (in 2022 dollars)  20 non-null     object
 5   Artist                            20 non-null     object
 6   Tour title                        20 non-null     object
 7   Year(s)                           20 non-null     object
 8   Shows                             20 non-null     int64 
 9   Average gross                     20 non-null     object
 10  Ref.                              20 non-null     object
dtypes: int64(2), object(9)
memory usage: 1.8+ KB


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   rank                            20 non-null     int64  
 1   actual_gross                    20 non-null     float64
 2   adjusted_gross_in_2022_dollars  20 non-null     float64
 3   artist                          20 non-null     object 
 4   tour_title                      20 non-null     object 
 5   years                           20 non-null     int64  
 6   shows                           20 non-null     int64  
 7   average_gross                   20 non-null     float64
 8   start_year                      20 non-null     int64  
 9   end_year                        20 non-null     int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 1.7+ KB


### This process is simple and useful, a good data structure is essential for data analysis and machine learning models. As we can see, the dataset is very small, but for larger projects many more transformations could be used. For this project i guess it was sufficient.